# PaddleOCR-VL-1.6 on the hardest FinTabNet tables (Colab)

Phase-A/C specialist ceiling run. Scores **PaddleOCR-VL-1.6 (0.9B)** — a table-SOTA
document parser — over the 50 hardest FinTabNet validation tables (merged cells
guaranteed), through the repo's existing TEDS-Struct + span-recall + bootstrap-CI
pipeline. GLM-OCR is added in a second notebook once this one is proven end-to-end.

**Public data only.** FinTabNet is public S&P-500 tables, so nothing here touches the
confidential-invoice boundary — Colab is fine.

**This is a debugging run, by design.** `paddle_client._extract_table_html` guesses the
PaddleOCR-VL result-object shape (the one open item flagged in CLAUDE.md). Cell 4
inspects the raw result on a single image *before* the full 50-table pass so you can
confirm/patch the extraction there instead of at the end.

**Prereq — push the branch first.** `paddle_client.py`, `api_client.py`, `images.py`
and the `inference.py` model constants must be committed and pushed, or the clone below
won't contain them:
```
git add src/ && git commit -m "phase-A specialist clients" && git push -u origin phase-two-distillation-pipeline
```
Runtime: **GPU** (T4 is plenty for 0.9B) — Runtime ▸ Change runtime type ▸ T4 GPU.

## 1. Setup — clone repo + install PaddleOCR-VL

`paddleocr`/`paddlepaddle` are deliberately **not** in `requirements-base.txt` (kept out
so `src/` imports without them on the Mac). We add them here. Versions may need nudging —
PaddleOCR-VL needs paddleocr 3.x on paddlepaddle 3.x; if `PaddleOCRVL` fails to import,
that's the first thing to bump.

In [1]:
BRANCH = "phase-two-distillation-pipeline"
REPO   = "https://github.com/hidrochin/qwen-vl-table-reconstruction.git"

import os, sys
if not os.path.isdir("qwen-vl-table-reconstruction"):
    !git clone --branch $BRANCH $REPO
%cd qwen-vl-table-reconstruction
!git checkout $BRANCH && git pull --ff-only

# Repo runtime deps (loader uses requests; eval uses numpy). CPU-only, installs fast.
!pip install -q -r requirements-base.txt
# Specialist pipeline — the two packages held out of requirements-base.
!pip install -q -U "paddlepaddle-gpu" "paddleocr"

# Make `import src...` work from the repo root.
sys.path.insert(0, os.getcwd())
print("\ncwd:", os.getcwd())

Cloning into 'qwen-vl-table-reconstruction'...
remote: Enumerating objects: 91, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 91 (delta 19), reused 84 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (91/91), 281.25 KiB | 15.63 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/qwen-vl-table-reconstruction
Already on 'phase-two-distillation-pipeline'
Your branch is up to date with 'origin/phase-two-distillation-pipeline'.
Already up to date.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 128.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.9/147.9 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 43.1 

## 2. Build the 50 hardest-table eval corpus

`build_split` scans the FinTabNet **validation** split, ranks by difficulty, keeps the
top 50 tables that contain at least one span (merged cell — the capability under test),
deduplicates by normalized HTML, and downloads only those images (~a few MB). Only HTML
is scored during the scan, so paging thousands of rows is cheap.

**Rate limit (the earlier failure).** The anonymous `datasets-server` API returns
**HTTP 429** under sustained paging — that's what killed the previous run at ~2,000 rows,
not a model error. Two mitigations are now in place:

1. `max_scanned=5000` below — scan a portion, not the whole 20k split. ~2k spanning
   candidates by then, so the hardest 50 are well-covered.
2. The loader now paces requests, honors `Retry-After`, and waits out a 429 instead of
   dying. Setting `HF_TOKEN` (see the cell) lifts the limit further.

Bump `max_scanned` (or set `HF_TOKEN`) if you see a `wanted 50, got N` warning.

In [ ]:
from pathlib import Path
from src.data.loader import build_split, load_manifest

# OPTIONAL but recommended: an HF token lifts the datasets-server rate limit a lot,
# so the scan won't throttle. Get one at https://huggingface.co/settings/tokens
# (read scope is enough) and uncomment:
# import os; os.environ["HF_TOKEN"] = "hf_xxx"

CORPUS = Path("data/corpus")
records = build_split(
    split_name="eval",
    source_split="validation",
    n_target=50,
    out_dir=CORPUS,
    # Scan a portion, not the whole split. The anonymous datasets-server API 429s
    # under sustained paging; 5,000 rows already yields ~2k spanning candidates, so
    # the hardest 50 are well-covered. Raise this (or set HF_TOKEN above) if you see
    # a "wanted 50, got N" warning.
    max_scanned=5000,
)
print(f"\nbuilt {len(records)} tables -> {CORPUS/'eval'}")

## 3. Load PaddleOCR-VL

First construction downloads the 0.9B weights. `PaddleTableReconstructor` wraps the
pipeline behind `predict`/`predict_many` so its output flows into the same eval path as
every other candidate.

In [ ]:
from src.model.paddle_client import PaddleTableReconstructor, _extract_table_html

reconstructor = PaddleTableReconstructor()
pipeline = reconstructor._get_pipeline()   # triggers the weight download
print("PaddleOCR-VL ready")

## 4. Debugging pass — confirm the result-object shape on ONE image

**This is the cell that de-risks the run.** `_extract_table_html` walks the pipeline
output looking for `<table>…</table>`. If it returns empty here, patch the extraction
(inspect `raw` below and adjust the attr list / traversal in `src/model/paddle_client.py`)
*before* spending time on all 50. Don't proceed to cell 5 until this prints a table.

In [ ]:
records = load_manifest(CORPUS / "eval")
sample = records[0]
print("image:", sample.image_path, "\n")

raw_result = pipeline.predict(sample.image_path)
print("type(raw_result):", type(raw_result))
print("repr (truncated):\n", repr(raw_result)[:1500], "\n")

extracted = _extract_table_html(raw_result)
print("=== extracted <table> (first 800 chars) ===")
print(extracted[:800] if extracted else "*** EMPTY — patch _extract_table_html before continuing ***")

## 5. Predict over all 50 tables

In [ ]:
image_paths = [r.image_path for r in records]
preds = reconstructor.predict_many(image_paths)
print(f"\n{len(preds)} predictions; "
      f"{sum(1 for p in preds if not p.html.strip())} empty")

## 6. Score + report

Same pipeline as the API bake-off: TEDS-Struct (structure-only), position-aware span
recall, bootstrap CIs, per-difficulty-bin breakdown, and parse failures. Remember these
specialists emit their **own HTML dialect** — read TEDS-Struct as a *ceiling reference*
("what a table-SOTA parser gets zero-shot"), and lean on **span recall** for the
customer-legible "recovered N of M merged cells" number.

In [3]:
from src.eval.runner import evaluate_predictions, save_run
from src.model.inference import predictions_dict

results, summary = evaluate_predictions(records, predictions_dict(preds), "paddleocr-vl-1.6")
save_run(results, summary, Path("outputs/runs"))

print(f"=== PaddleOCR-VL-1.6 over {summary.n} hardest FinTabNet tables ===")
print(f"TEDS-Struct : {summary.mean_teds:.4f}  [{summary.ci_low:.3f}, {summary.ci_high:.3f}]")
print(f"span-recall : {summary.mean_span_recall:.4f}")
print(f"parse-fails : {summary.parse_failures}/{summary.n}")
if summary.by_bin:
    print("\nby difficulty bin:")
    for label, b in summary.by_bin.items():
        print(f"  {label:<7} n={b['n']:<3} TEDS {b['mean']:.4f} [{b['ci'][0]:.3f}, {b['ci'][1]:.3f}]")

NameError: name 'records' is not defined

## 7. Eyeball the hardest failures

Worst TEDS-Struct cases first — this is where dialect-vs-real-error gets sorted out.

In [ ]:
from src.eval.runner import to_cases

for c in to_cases(results, limit=3, hardest_first=True):
    print("=" * 70)
    print(f"{c.uid}  TEDS-Struct={c.score:.3f}  difficulty={c.difficulty}  spans={c.n_spanning}")
    print("  image:", c.image_path)
    print("  --- predicted (first 400 chars) ---\n ", c.pred_html[:400])
    print("  --- ground truth (first 400 chars) ---\n ", c.true_html[:400])


## Next: GLM-OCR

Once PaddleOCR-VL is proven end-to-end here, GLM-OCR (0.9B, MIT) gets a sibling client
`src/model/glm_client.py` with the same `predict`/`predict_many` surface, and a second
notebook (or an added cell here) that scores it over the **same** `data/corpus/eval`
records and runs `compare_runs(paddle_summary, glm_summary)` for the head-to-head.